# Machine learning training

------------------------------------------

### Initializing the work environment

```bash
python -m venv venv
source venv/bin/activate
pip install -r requirement.txt
```

### Import the necessary libs

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans
from sklearn.cluster import DBSCAN
from sklearn.decomposition import NMF
from sklearn.metrics import silhouette_score
from sklearn.decomposition import TruncatedSVD
from sklearn.cluster import AgglomerativeClustering
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS

-------------------------------------------------------------

The dataset consists of 3 columns: ID, a secret color, and the testimony. The color will be used to check if our clusters are accurate at the end of the learning phase of our project.

Our objective is to group similar testimonies into clusters, where each cluster potentially corresponds to a specific diagnosis or underlying disease.

In [ ]:
def init(path):
    df = pd.read_csv(path, header=None, names=[0, 1, 2])
    print(f"dataset load perfectly : {df.shape}")
    return df

df = init("./../non-supervisé/cleaned_dataset.csv")

In [ ]:
# function for print the dataset (debug or visualization purpose)
def print_df(n, df):
    df_print = df.head(n)
    print(f"this is the {n} first lines of the dataset:\n\n{df_print}")

----------------------------------------

## Training with cleaned dataset

### First step : Vectorize the dataset 

In [ ]:
# This function will vectorize, meaning it will put the words we are truly interested in into a matrix and assign them a value based on their presumed importance in the dataset
def vectorize_text(df):
    # This line configures the rules to define the importance of the words so that the training can proceed correctly
    vectorizer = TfidfVectorizer(stop_words='english')
    # This line will create the matrix and transform the text into values according to the TF-IDF protocol
    X_tfidf = vectorizer.fit_transform(df[2])
    svd = TruncatedSVD(n_components=50, random_state=42)
    X_svd = svd.fit_transform(X_tfidf)
    
    # This line will be used to debug and check the words that were kept
    kept_words = vectorizer.get_feature_names_out()
    # Visualizing the words kept by our code
    print(f"Number of highly relevant medical words kept: {len(kept_words)}")

    return X_tfidf, X_svd, vectorizer, svd

X_tfidf, X_svd, vectorizer, svd = vectorize_text(df)

Okay, now we are going to show the useful words kept by the function and try to understand how to find the different clusters.

Extremely simplified example of matrix X :

|  | Col n0 : "ache" | Col n1 : "chest" |
|:-------- |:--------:| --------:|
| Patient 1| 0.0   | 0.2    |
| Patient 2| 0.4   | 0.1    |

In [ ]:
# X: TF-IDF matrix, vectorizer: The fitted TfidfVectorizer object (our vocabulary/dictionary)
def plot_top_tfidf_words(X, vectorizer, top_n=20):

    # This line calculates the sum of all values in each column (i.e., the total TF-IDF score for each word across all documents)
    tfidf_sum = np.array(X.sum(axis=0)).flatten()
    # This line retrieves the actual readable words (feature names) corresponding to the matrix columns
    words = vectorizer.get_feature_names_out()
    # This sorts the array and extracts the indices of the top 20 words with the highest overall scores
    top_indices = tfidf_sum.argsort()[-top_n:]
    
    # This extracts the actual words and their corresponding scores using the sorted indices
    top_words = [words[i] for i in top_indices]
    top_scores = tfidf_sum[top_indices]
    
    plt.figure(figsize=(10, 8))
    plt.barh(top_words, top_scores, color='coral')
    plt.title(f"Top {top_n} most 'discriminant' words")
    plt.xlabel("Score")
    plt.ylabel("Medical words")
    plt.grid(axis='x', linestyle='--', alpha=0.7)
    plt.show()

plot_top_tfidf_words(X_tfidf, vectorizer)

Memo english word to french :
- sore = Endolori (qui fait mal)
- ache = Douleur
- cough = Toux
- sharp = Aigu / Vif

Interpretation of the "most discriminative words":

We have some body parts, which will be very useful to give a diagnosis. The body parts we have are: chest, skin, ear, and throat.
We also have some feelings (symptoms) about them: pain (ache), cold, sharp, long, sore, and cough (cough is not a feeling, but a good clue to give a diagnosis).
So, we can now try to understand and think about a way to train an AI to give a diagnosis based on a testimony. For example: if the testimony has "cough" and "throat" in it, it could be tonsillitis (maux de gorge).

---------------------------------

## Cloud Analysis and cluster reflection

Now, we will see how to group the different lines of our dataset into clusters and determine the optimal number of clusters we need.

### First step : Try with the K-means machine learning algorithm

-------------------------------------

The Elbow Method helps us find the perfect number of clusters. We test the algorithm with 1, 2, 3... up to 12 clusters and calculate the "inertia" (which measures how tight our groups are). Then, we plot these values on a graph.

The curve looks like an arm. The best number of clusters is exactly at the "elbow", where the line stops dropping fast and becomes almost flat. Adding more clusters after this point doesn't improve the result.

In [ ]:
def find_optimal_k_combined(X, max_k=20):
    inertias = []
    scores = [] # Only for K >= 2

    K_range = range(1, max_k + 1)
    
    print("Calculating scores...")
    
    for k in K_range:
        kmeans = KMeans(n_clusters=k, random_state=42, n_init='auto')
        kmeans.fit(X)
        inertias.append(kmeans.inertia_)
        
        # We only calculate silhouette score if K > 1
        if k > 1:
            score = silhouette_score(X, kmeans.labels_)
            scores.append(score)
            
    # dual axis creation
    fig, ax1 = plt.subplots(figsize=(15, 8))

    # Left axis : INERTIA (Elbow Method)
    color1 = 'teal'
    ax1.set_xlabel('Number of clusters (K)', fontsize=14)
    ax1.set_ylabel('Inertia (Intra-cluster distance)', color=color1, fontsize=14)
    ax1.plot(K_range, inertias, marker='o', linestyle='-', color=color1, markersize=8, label="Inertia (Elbow)")
    ax1.tick_params(axis='y', labelcolor=color1)
    ax1.set_xticks(K_range)
    ax1.grid(True, linestyle='--', alpha=0.5)

    # Right axis : SILHOUETTE
    ax2 = ax1.twinx() 
    color2 = 'purple'
    ax2.set_ylabel('Silhouette Score (Higher is better)', color=color2, fontsize=14)
    
    # Note: 'scores' list has one less value than 'inertias' (no K=1)
    K_range_sil = range(2, max_k + 1)
    ax2.plot(K_range_sil, scores, marker='s', linestyle='--', color=color2, markersize=8, label="Silhouette Score")
    ax2.tick_params(axis='y', labelcolor=color2)

    plt.title("Combined Metrics: Elbow Method & Silhouette Score", fontsize=16, weight='bold')
    
    fig.tight_layout()
    plt.show()

find_optimal_k_combined(X_svd, max_k=20)

To interpret the Elbow Method graph, we must identify a significant change in the slope of the curve. Here, we can observe a very clear 'elbow' at K=10 and K=8 and K=18, which gives us the optimal number of clusters for our dataset.
I decided to add another metric to base our judgment on: the Silhouette score. In theory, the higher the Silhouette score, the better the clusters. Therefore, if we hesitate between two potential elbows, we can cross-reference with this score to make the best choice. In this case, we can clearly see that K=10 forms a distinct elbow and simultaneously has the best Silhouette score.

As we can observe, we do not simply select the lowest possible inertia. We must find a balance between a low inertia and a reasonable number of clusters. For instance, creating 1,000 clusters for 1,000 patients would result in an inertia of 0, but this would lead to overfitting and defeat the whole purpose of clustering.

In [ ]:
def apply_kmeans(X, n_clusters):
    # 1. K-Means Training
    kmeans = KMeans(n_clusters=n_clusters, random_state=42)
    kmeans.fit(X)
    labels_predict = kmeans.labels_
    
    # 2. Reduction with t-SNE
    tsne = TSNE(n_components=2, random_state=42, perplexity=30)
    X_tsne = tsne.fit_transform(X) 
    
    plt.figure(figsize=(15, 8))
    plt.tricontourf(X_tsne[:, 0], X_tsne[:, 1], labels_predict, cmap='tab10', alpha=0.2)
    plt.scatter(X_tsne[:, 0], X_tsne[:, 1], c=labels_predict, cmap='tab10', alpha=0.6, s=50)
    
    for i in range(n_clusters):
        cluster_points = X_tsne[labels_predict == i]
        
        # Calculate the center (mean of X and mean of Y)
        center_x = cluster_points[:, 0].mean()
        center_y = cluster_points[:, 1].mean()
        
        # Draw a large cross with a black border
        plt.scatter(center_x, center_y, c='black', marker='X', s=200, edgecolors='black', linewidths=1, zorder=2)

    plt.title("Cloud Point")
    plt.xlabel("t-SNE Dimension 1")
    plt.ylabel("t-SNE Dimension 2")
    plt.grid(True, linestyle='--', alpha=0.3)
    plt.show()
    
    return kmeans

In [ ]:
kmeans = apply_kmeans(X_svd, n_clusters=10)

def display_top_words_per_cluster(kmeans, X_tfidf, X_svd, vectorizer, n_mots=5):
    # We retrieve the model's "centers of gravity" (centroids) and our vocabulary/dictionary
    words = vectorizer.get_feature_names_out()
    labels = kmeans.labels_
    score = silhouette_score(X_svd, labels)
    print(f"Silhouette score : {score}")

    X_dense = X_tfidf.toarray()
    for i in range(kmeans.n_clusters):
        patients = X_dense[labels == i]
        if len(patients) > 0:
            mean_word = patients.mean(axis=0)
            top_indices = mean_word.argsort()[-n_mots:][::-1]
            top_words = [words[ind] for ind in top_indices]
            print(f"Cluster {i} ({len(patients)} patients) : {', '.join(top_words)}")

# We call the function to display the top 6 defining words for each disease cluster
display_top_words_per_cluster(kmeans, X_tfidf, X_svd, vectorizer, n_mots=6)

Here, after the learning algorithm identifies the optimal clusters, we reduce the dimensionality of our data to 2 parameters (2D) to plot it on a graph. However, as we can see, it is hard to tell which points belong together. To fix this, we will color the points into different clusters based on their similarities and add a cross which is represent the center of the clusters we have.

As we can observe on the scatter plot, the result with 10 clusters is quite good. Nearly all points of the same color stay closely grouped together. While natural noise remains, it is clearly a highly satisfactory result. Furthermore, if we analyze the most discriminant words for each group, we can see that they perfectly describe distinct medical conditions. This proves that our clusters are not only mathematically robust, but they also make perfect clinical sense.

Here is the medical interpretation of the 10 identified clusters:

- Cluster 0 - Upper Body Orthopedics (shoulder, arm, elbow, lifting): Highlights joint and muscle pain localized in the upper limbs.

- Cluster 1 - Lower Body Orthopedics (knee, leg, hip, deep ache): Groups lower limb joint pain and mobility issues.

- Cluster 2 - Diabetes / Metabolic Issues (blood, thirsty, weight, losing, sugar): A textbook description of diabetes symptoms, including severe thirst and sudden weight loss.

- Cluster 3 - Orthostatic Hypotension / Anemia (pale, dizzy, stand, headaches): Captures the specific sensation of dizziness and paleness when standing up.

- Cluster 4 - Ear Infection / Otitis (ear, pain, fever, hear): Groups symptoms related to earaches, fever, and mild hearing difficulties.

- Cluster 5 - Localized Skin Inflammation (hot, red, sore, area): Describes acute, localized skin issues such as abscesses, burns, or local infections.

- Cluster 6 - Mild Cold / Viral Infection (nose, throat, mild, cough): Represents common, mild upper respiratory tract infections.

- Cluster 7 - Migraine (head, throbbing, vision): Perfectly isolates severe, throbbing headaches associated with visual disturbances.

- Cluster 8 - Asthma / Respiratory Distress (chest, short breath, tightness): Highlights severe lower respiratory issues, chest tightness, and breathing difficulties.

- Cluster 9 - Allergic Reaction (itchy, rash, hives, spots): Groups dermatological allergies, rashes, and urticaria (hives).

### Second step : Try with the DBScan machine learning algorithm

-------------------------------------

Now, we will try applying the DBSCAN algorithm to our dataset. DBSCAN is a highly unsuitable model for this specific task because it looks for dense clusters that are clearly separated from one another by empty spaces. As we have seen with our K-Means results, our textual data is continuous and does not look like that at all. However, it is very instructive to observe how an algorithm behaves when its mathematical logic is not adapted to the nature of the data.

In [ ]:
def apply_dbscan(X_svd):
    print("1. Launching DBSCAN...")
    # eps and min_samples are values to help DBSCAN find clusters
    # eps is the search radius around a single patient 
    # min_samples is the number of patients needed to form a cluster
    dbscan = DBSCAN(eps=0.5, min_samples=5)
    labels_dbscan = dbscan.fit_predict(X_svd)
    
    # Here we figure out how many clusters DBSCAN found. If label -1 exists (noise), it doesn't count as a cluster
    n_clusters = len(set(labels_dbscan)) - (1 if -1 in labels_dbscan else 0)
    n_noise = list(labels_dbscan).count(-1)
    total_patients = len(labels_dbscan)
    
    print(f"Number of clusters found: {n_clusters}")
    print(f"Noise: {n_noise} / {total_patients} ({(n_noise/total_patients)*100:.2f}%)")
    
    # Here we count the number of patients in each found cluster
    for i in range(n_clusters):
        print(f"Cluster {i} size: {list(labels_dbscan).count(i)} patients")

    tsne = TSNE(n_components=2, random_state=42, perplexity=30)
    X_tsne = tsne.fit_transform(X_svd)
    
    plt.figure(figsize=(12, 8))
    scatter = plt.scatter(X_tsne[:, 0], X_tsne[:, 1], c=labels_dbscan, cmap='tab20', alpha=0.7, s=50)
    plt.title("DBSCAN Failure: High density of Noise (-1) or Giant Cluster")
    plt.xlabel("Dimension 1")
    plt.ylabel("Dimension 2")
    plt.show()

apply_dbscan(X_svd)

Unlike K-Means, DBSCAN attempts to find clusters based on data density rather than distance to a central point. Consequently, we cannot force it to generate a specific number of clusters. This makes it completely unsuitable for our situation. As we can see above, it only found 2 clusters. Therefore, there is no reason to explore this algorithm any further for this specific subject.

### Third step : Try with the Agglomerative machine learning algorithm

---------------------------------------------------

In [ ]:
def find_optimal_k_agglo_svd(X_svd, max_k=20):
    # This list will store the silhouette score calculated for each k
    silhouette_scores = []
    
    # We must start at 2, because silhouette score requires at least 2 clusters
    K_range = range(2, max_k + 1)
    
    for k in K_range:
        agg_model = AgglomerativeClustering(n_clusters=k, linkage='ward')
        
        # X_svd is ALREADY a dense numpy array, so we removed the .toarray() method!
        labels = agg_model.fit_predict(X_svd)
        
        # We calculate the silhouette score on the reduced SVD space
        score = silhouette_score(X_svd, labels)
        silhouette_scores.append(score)
    
    # --- GRAPHIC GENERATION ---
    plt.figure(figsize=(15, 8))
    plt.plot(K_range, silhouette_scores, marker='o', linestyle='-', color='purple', markersize=8)
    
    plt.title("Silhouette Method to find the optimal K (Agglomerative on SVD)")
    plt.xlabel("Number of clusters (K)")
    plt.ylabel("Silhouette Score (HIGHER is better)")
    plt.xticks(K_range)
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.show()

# On lance la fonction en lui donnant bien la variable X_svd générée par ta fonction précédente !
find_optimal_k_agglo_svd(X_svd, max_k=20)

This graph displays the Silhouette score for the Agglomerative Clustering algorithm, tested across a range of 2 to 20 clusters. This metric is crucial for identifying the most efficient cluster count. Although the Silhouette score is fundamentally a theoretical mathematical indicator, it serves as our most robust evaluation method, especially considering the strict library constraints we had to work with for this model.

In [ ]:
def apply_agg(X_svd, n_clusters=12):
    agg_model = AgglomerativeClustering(n_clusters=n_clusters, linkage='ward')
    labels_agg = agg_model.fit_predict(X_svd)
    tsne = TSNE(n_components=2, random_state=42, perplexity=30)
    X_tsne = tsne.fit_transform(X_svd)
    plt.figure(figsize=(15, 8))
    # plt.tricontourf(X_tsne[:, 0], X_tsne[:, 1], labels_agg, cmap='tab10', alpha=0.2)
    plt.scatter(X_tsne[:, 0], X_tsne[:, 1], c=labels_agg, cmap='tab20', alpha=0.6, s=50) # tab20 car tu as 16 clusters !
    
    for i in range(n_clusters):
        points_du_cluster = X_tsne[labels_agg == i]
        if len(points_du_cluster) > 0:
            centre_x = points_du_cluster[:, 0].mean()
            centre_y = points_du_cluster[:, 1].mean()
            plt.scatter(centre_x, centre_y, c='red', marker='X', s=400, edgecolors='black', linewidths=2, zorder=5)

    plt.title(f"Archipelago of Diseases (Agglomerative LSA : SVD + t-SNE)")
    plt.xlabel("t-SNE Dimension 1")
    plt.ylabel("t-SNE Dimension 2")
    plt.grid(True, linestyle='--', alpha=0.3)
    plt.show()

    return labels_agg

In [ ]:
labels_agg = apply_agg(X_svd)

def display_top_words_per_cluster_alggo(labels_agg, X_tfidf, n_clusters=12):
    mots_gardes = vectorizer.get_feature_names_out()
    X_dense = X_tfidf.toarray()
    score = silhouette_score(X_svd, labels_agg)
    print(f"Silhouette score : {score}")
    
    for i in range(n_clusters):
        patients_du_cluster = X_dense[labels_agg == i]
        if len(patients_du_cluster) > 0:
            moyenne_mots = patients_du_cluster.mean(axis=0)
            top_mots_idx = moyenne_mots.argsort()[-6:][::-1]
            top_mots = [mots_gardes[idx] for idx in top_mots_idx]
            print(f"Cluster {i} ({len(patients_du_cluster)} patients) : {', '.join(top_mots)}")

display_top_words_per_cluster_alggo(labels_agg, X_tfidf)

As we can observe with this model, the data points are no longer dispersed; they form very distinct and cohesive clusters. When we look at the most discriminant words for each group, we can clearly see that the categorization of each patient's disease is even more accurate and refined than with the K-Means model. Here is the medical interpretation of these 12 specific clusters:

- Cluster 0 - Cardiopulmonary Emergency (chest, cough, breath, pressure, short, tight): Highlights severe respiratory distress, asthma attacks, or potential cardiac issues. Interestingly, this is the largest group, which mirrors real-life emergency room volumes.

- Cluster 1 - Diabetes / Metabolic (vision, doctor, blood, losing, sugar, weight): A perfect textbook description of diabetes symptoms (blood sugar issues, vision changes, sudden weight loss).

- Cluster 2 - Severe Headache / Migraine (pain, head, starts, throbbing, ache, dull): Focuses on cranial pain, specifically throbbing or dull headaches.

- Cluster 3 - Lower Body Orthopedics (knee, deep, ache, leg, thigh, hip): Groups major joint and muscle pain in the lower limbs (hips, thighs, knees).

- Cluster 4 - Mild Viral Infection (nose, throat, mild, cough, day, cold): Represents benign, mild upper respiratory tract infections (the common cold).

- Cluster 5 - Localized Skin Infection / Burn (hot, blister, touched, small, red, skin): Clearly describes acute skin lesions, such as blisters, burns, or local infections.

- Cluster 6 - Ear Infection / Otitis (ear, pain, slight, fever, cold, hear): Specifically isolates ENT issues related to earaches and minor hearing loss.

- Cluster 7 - Allergic Reaction (itchy, rash, new, spots, raised, hives): Perfectly groups dermatological allergies, including rashes and hives.

- Cluster 8 - Distal Upper Limb Orthopedics (wrist, elbow, palm, thumb, forearm): Highly specific cluster focusing on distal arm issues (tendinitis, carpal tunnel).

- Cluster 9 - Vagal Syncope / Hypotension (pale, dizzy, skin, stand, breath, heart): Captures the exact sensation of dizziness, paleness, and heart palpitations when standing up.

- Cluster 10 - Foot & Ankle Orthopedics / Podiatry (foot, toe, pushing, shoes, big, ankle): The most precise orthopedic cluster, focusing strictly on extremities and footwear-related pain.

- Cluster 11 - Proximal Upper Limb Orthopedics (shoulder, reaching, arm, overhead, lifting, sharp): Focuses on shoulder joint issues and pain triggered by lifting or overhead movements.

--------------------------------

## Conclusion

In conclusion, we have identified two viable models for clustering the multiple diseases that hospital patients present. However, the Agglomerative Clustering model proved to be the most effective and powerful. This is because the agglomerative approach can cluster closely related data more easily and with greater precision than K-Means. At the end of this unsupervised learning phase, we successfully isolated 12 distinct clusters, each representing a specific medical condition. Moving forward, incoming patients can be automatically evaluated against these clusters and directly routed to the appropriate hospital departments.

---------------------------------------------

## Bonus model

In [ ]:
import joblib
from sklearn.ensemble import RandomForestClassifier

classifieur = RandomForestClassifier(random_state=42)
classifieur.fit(X_svd, labels_agg)

joblib.dump(vectorizer, 'vectoriseur_kmeans.pkl')
joblib.dump(svd, 'svd_kmeans.pkl')
joblib.dump(kmeans, 'kmeans_cvrie.pkl')

For lunching the application make sure to type the following lines (after lunching this notebook) :

```bash
cd bonus
streamlit run app.py
```